In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

from sklearn.linear_model import LassoCV, ridge_regression
from sklearn.metrics import mean_absolute_percentage_error, r2_score

In [4]:
# --- Columns ---
y_col = "total_transmit_duration"
x_size = "total_transmit_size"
x_count = "transmit_count"

def visualize(train_df: pd.DataFrame, model: sm.regression.linear_model.RegressionModel):
    # --- Build a prediction grid over size × count ---
    n_side = 50  # grid resolution per axis (increase for smoother surface)
    size_lin = np.linspace(train_df[x_size].min(), train_df[x_size].max(), n_side)
    count_lin = np.linspace(train_df[x_count].min(), train_df[x_count].max(), n_side)
    S, C = np.meshgrid(size_lin, count_lin)

    grid_df = pd.DataFrame({x_size: S.ravel(), x_count: C.ravel()})

    # Mean prediction
    mean_pred = model.predict(grid_df)
    Z_mean = mean_pred.reshape(S.shape)

    # # 95% Prediction Interval (for new observations)
    # Z_mean = mean_pred["mean"].values.reshape(S.shape)
    # Z_pi_low = mean_pred["obs_ci_lower"].values.reshape(S.shape)
    # Z_pi_high = mean_pred["obs_ci_upper"].values.reshape(S.shape)

    # --- Build interactive 3D figure ---
    fig = go.Figure()

    # Fitted surface (mean)
    fig.add_trace(go.Surface(x=S, y=C, z=Z_mean, name="Fitted surface (mean)", showscale=False, opacity=0.85))

    # Raw data points
    fig.add_trace(
        go.Scatter3d(
            x=train_df[x_size],
            y=train_df[x_count],
            z=train_df[y_col],
            mode="markers",
            name="Data",
            marker=dict(size=3, opacity=0.6),
        )
    )

    # Optional: lower PI surface (toggle via legend)
    # fig.add_trace(
    #     go.Surface(x=S, y=C, z=Z_pi_low, name="Lower 95% PI", showscale=False, opacity=0.25, visible="legendonly")
    # )

    # # Optional: upper PI surface (toggle via legend)
    # fig.add_trace(
    #     go.Surface(x=S, y=C, z=Z_pi_high, name="Upper 95% PI", showscale=False, opacity=0.25, visible="legendonly")
    # )

    fig.update_layout(
        title="Interactive 3D: Duration ~ Size + Transmit Count",
        scene=dict(
            xaxis_title="Total Transmit Size",
            yaxis_title="Transmit Count",
            zaxis_title="Total Transmit Duration",
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),
        ),
        legend=dict(itemsizing="constant"),
    )

    fig.show()

In [5]:
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import Pipeline


def evaluate(y_true, y_pred, X):
    print("R² =", r2_score(y_true, y_pred))
    print("MAPE =", mean_absolute_percentage_error(y_true, y_pred))

    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true)))
    print(f"SMAPE = {smape:.2f}%")

    # --- Variance normalization (bin-based) ---
    X = X.values
    n_bins = 8
    bins = [np.linspace(X[:, i].min(), X[:, i].max(), n_bins + 1) for i in range(X.shape[1])]
    idx = [np.digitize(X[:, i], bins[i]) - 1 for i in range(X.shape[1])]
    sigma = np.zeros_like(y_true)

    # --- MacroMAPE (uniform over X space) ---
    mapes = []
    for b1 in range(n_bins):
        for b2 in ([0] if X.shape[1] == 1 else range(n_bins)):
            mask = (idx[0] == b1) & (idx[-1] == b2)
            if np.sum(mask) > 0:
                mapes.append(mean_absolute_percentage_error(y_true[mask], y_pred[mask]))
    macro_mape = np.mean(mapes)
    print(f"MacroMAPE = {macro_mape:.4f}")


def fit_model(train_df: pd.DataFrame):
    X = train_df[[x_size, x_count]].copy()
    y = train_df[y_col].copy()

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LassoCV(cv=5, fit_intercept=True))  # drop positive=True unless justified
    ]).fit(X, y)

    coefs = pd.Series(model.named_steps['model'].coef_, index=X.columns)
    print(f"Scaling: {model.named_steps['scaler'].scale_}")

    coefs_unscaled = coefs / model.named_steps['scaler'].scale_
    print("Coefficients:")
    for name, val in coefs_unscaled.items():
        print(f"  {name} = {val:.6e}")
    if hasattr(model.named_steps['model'], "intercept_"):
        print(f"Intercept = {model.named_steps['model'].intercept_:.6e}")

    # model = LassoCV(cv=5, positive=True, fit_intercept=False).fit(X, y)
    # model = sm.OLS(y, X).fit()

    y_pred = model.predict(X)

    print(model.get_params())
    # print(model.params)

    # Predictions
    y_pred = model.predict(X) 
    evaluate(y, y_pred, X)

    return model

In [6]:
import json

dfs = []

for i in [2, 4, 8]:
    with open(f'../results/quest/tp_nccl_{i}_0.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"] * (i - 1)
    df.loc[:, "transmit_count"] = df["transmit_count"] * (i - 1)
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"] 
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

print(df.size)

model = fit_model(df)

X = df[[x_size, x_count]]
y = df[y_col]
y_pred = model.predict(X)

evaluate(y, y_pred, X)
visualize(df, model)

1134
Scaling: [2.92713343e+08 3.73250801e+03]
Coefficients:
  total_transmit_size = 1.228575e-11
  transmit_count = 4.725710e-05
Intercept = 1.875902e-01
{'memory': None, 'steps': [('scaler', StandardScaler()), ('model', LassoCV(cv=5))], 'transform_input': None, 'verbose': False, 'scaler': StandardScaler(), 'model': LassoCV(cv=5), 'scaler__copy': True, 'scaler__with_mean': True, 'scaler__with_std': True, 'model__alphas': 'warn', 'model__copy_X': True, 'model__cv': 5, 'model__eps': 0.001, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__n_alphas': 'deprecated', 'model__n_jobs': None, 'model__positive': False, 'model__precompute': 'auto', 'model__random_state': None, 'model__selection': 'cyclic', 'model__tol': 0.0001, 'model__verbose': False}
R² = 0.9143693697504133
MAPE = 0.6055692926306955
SMAPE = 37.28%
MacroMAPE = 0.2160
R² = 0.9143693697504133
MAPE = 0.6055692926306955
SMAPE = 37.28%
MacroMAPE = 0.2160


In [7]:
dfs = []

for i in [4, 8]:
    with open(f'../results/quest/tp_gloo_{i}.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"] * (i - 1)
    df.loc[:, "transmit_count"] = df["transmit_count"] * (i - 1)
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

model = fit_model(df)

X = df[[x_size, x_count]]
y = df[y_col]
y_pred = model.predict(X)

evaluate(y, y_pred, X)
visualize(df, model)


756
Scaling: [3.31114884e+08 4.23702087e+03]
Coefficients:
  total_transmit_size = 3.261422e-09
  transmit_count = 5.616651e-04
Intercept = 3.741357e+00
{'memory': None, 'steps': [('scaler', StandardScaler()), ('model', LassoCV(cv=5))], 'transform_input': None, 'verbose': False, 'scaler': StandardScaler(), 'model': LassoCV(cv=5), 'scaler__copy': True, 'scaler__with_mean': True, 'scaler__with_std': True, 'model__alphas': 'warn', 'model__copy_X': True, 'model__cv': 5, 'model__eps': 0.001, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__n_alphas': 'deprecated', 'model__n_jobs': None, 'model__positive': False, 'model__precompute': 'auto', 'model__random_state': None, 'model__selection': 'cyclic', 'model__tol': 0.0001, 'model__verbose': False}
R² = 0.9597421850888116
MAPE = 0.32350690986350283
SMAPE = 23.87%
MacroMAPE = 0.1448
R² = 0.9597421850888116
MAPE = 0.32350690986350283
SMAPE = 23.87%
MacroMAPE = 0.1448


In [28]:
from sklearn.linear_model import LinearRegression

dfs = []

for i in [4, 8]:
    with open(f'../results/quest/tp_nccl_{i}_0.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"] * (i - 1)
    df.loc[:, "transmit_count"] = df["transmit_count"] * (i - 1)
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

X = df[[x_size, x_count]].copy()
y = df[y_col].copy()

def manual_fit(a = None, b = None):
    model = LinearRegression()
    smallest_loc = df[x_size] == df[x_size].min()
    if a is None:
        a = (y[smallest_loc] / X[smallest_loc][x_count]).mean()
    print(f"Anticipated latency per transmit (a): {a*1000:.4f} ms")
    bigger_loc = df[x_size] >= df[x_size].max() * 0.1
    if b is None:
        b = (y[bigger_loc] / X[bigger_loc][x_size]).mean()
    print(f"Anticipated Bandwidth: {1/(b * 1e6):.4f} MB/s")
    model.coef_ = np.array([b, a])
    model.feature_names_in_ = np.array(["total_transmit_size", "transmit_count"])
    model.intercept_ = 0
    return model

model = manual_fit() # 0.1503 * 1e-3, 1e-6/(1094.9022))

y_pred = model.predict(X)
y_true = y

evaluate(y_true, y_pred, X)

visualize(df, model)

Anticipated latency per transmit (a): 0.1323 ms
Anticipated Bandwidth: 1132.2649 MB/s
R² = -9.841742778626482
MAPE = 2.7440834978852884
SMAPE = 104.75%
MacroMAPE = 3.3047


In [98]:
from sklearn.linear_model import LinearRegression

dfs = []

for i in [8]:
    with open(f'../results/quest/tp2_gloo_{i}_0.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"]
    df.loc[:, "transmit_count"] = df["transmit_count"]
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

X = df[[x_size, x_count]].copy()
y = df[y_col].copy()

def manual_fit(a = None, b = None):
    model = LinearRegression()
    smallest_loc = df[x_size] == df[x_size].min()
    if a is None:
        a = (y[smallest_loc] / X[smallest_loc][x_count]).mean()
    print(f"Anticipated latency per transmit (a): {a*1000:.4f} ms")
    bigger_loc = df[x_size] >= df[x_size].max() * 0.1
    if b is None:
        b = (y[bigger_loc] / X[bigger_loc][x_size]).mean()
    print(f"Anticipated Bandwidth: {1/(b * 1e6):.4f} MB/s")
    model.coef_ = np.array([b, a])
    model.feature_names_in_ = np.array(["total_transmit_size", "transmit_count"])
    model.intercept_ = 0
    return model

model = manual_fit(4 * 0.4387 * 1e-3, 1e-6/(113.1185))

y_pred = model.predict(X)
y_true = y

evaluate(y_true, y_pred, X)

visualize(df, model)

Anticipated latency per transmit (a): 1.7548 ms
Anticipated Bandwidth: 113.1185 MB/s
R² = 0.2889595681623748
MAPE = 0.43689688463818516
SMAPE = 51.75%
MacroMAPE = 0.3875


In [99]:
from sklearn.linear_model import LinearRegression

dfs = []

for i in [4]:
    with open(f'../results/quest/tp2_nccl_{i}_0.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"]
    df.loc[:, "transmit_count"] = df["transmit_count"]
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

X = df[[x_size, x_count]].copy()
y = df[y_col].copy()

def manual_fit(a = None, b = None):
    model = LinearRegression()
    smallest_loc = df[x_size] <= df[x_size].min() * 10
    if a is None:
        a = (y[smallest_loc] / X[smallest_loc][x_count]).mean()
    print(f"Anticipated latency per transmit (a): {a*1000:.4f} ms")
    bigger_loc = df[x_size] >= df[x_size].max() * 0.99
    if b is None:
        b = (y[bigger_loc] / X[bigger_loc][x_size]).mean()
    print(f"Anticipated Bandwidth: {1/(b * 1e6):.4f} MB/s")
    model.coef_ = np.array([b, a])
    model.feature_names_in_ = np.array(["total_transmit_size", "transmit_count"])
    model.intercept_ = 0
    return model

model = manual_fit(1 * 0.2 * 1e-3, 1e-6/(1 * 12500))

y_pred = model.predict(X)
y_true = y

evaluate(y_true, y_pred, X)

visualize(df, model)

Anticipated latency per transmit (a): 0.2000 ms
Anticipated Bandwidth: 12500.0000 MB/s
R² = 0.795747309726441
MAPE = 0.33644319133426326
SMAPE = 25.05%
MacroMAPE = 0.3626
